# Actividad 3 | Aprendizaje supervisado y no supervisado
---
- César Iván Pedrero Martínez   |   A01366501

## 1. Introducción teórica: Aprendizaje supervisado y no supervisado

---

### Aprendizaje Supervisado

El aprendizaje supervisado es una técnica de aprendizaje automático en la que se entrena un modelo utilizando un conjunto de datos previamente etiquetado. Esto significa que cada ejemplo en los datos de entrenamiento está compuesto por una entrada (características) y una salida deseada (etiqueta). El objetivo principal es que el modelo aprenda a generalizar correctamente sobre nuevos datos.

Entre los algoritmos más representativos se encuentran:

* **Regresión lineal**: Predice valores continuos, como precios o temperaturas.
* **Regresión logística**: Utilizada en clasificación binaria (por ejemplo, clasificar correos como spam o no spam).
* **Árboles de decisión**: Modelo basado en reglas para clasificar o predecir salidas.
* **Máquinas de vectores de soporte (SVM)**: Eficaces en espacios de alta dimensión.
* **Bosques aleatorios (Random Forest)**: Ensembles de árboles que mejoran rendimiento y evitan el sobreajuste.

Estos algoritmos se aplican en tareas como diagnóstico médico, predicción financiera, y reconocimiento de voz o imágenes.

---

### Aprendizaje No Supervisado

El aprendizaje no supervisado se aplica cuando se dispone de datos no etiquetados. Su propósito es identificar patrones, estructuras o agrupamientos latentes sin una referencia explícita de salida. Este enfoque es útil para explorar datos y obtener conocimiento no evidente a simple vista.

Entre los algoritmos más comunes destacan:

* **K-means**: Agrupa datos en clústeres con base en similitud.
* **BisectingKMeans**: Variante jerárquica de K-means que mejora la precisión del agrupamiento.
* **GaussianMixture**: Modelo de mezcla de gaussianas que permite representar clústeres con formas elípticas y superpuestas, útil cuando se espera que los datos provengan de distribuciones normales múltiples.
* **PCA (Análisis de Componentes Principales)**: Técnica para reducir la dimensionalidad del conjunto de datos conservando la mayor cantidad de varianza posible.

Estos algoritmos permiten, entre otras cosas, realizar segmentación de clientes, detección de anomalías, compresión de datos y análisis exploratorio de grandes volúmenes de información.

---

### Combinando técnicas

Es posible combinar técnicas de aprendizaje supervisado y no supervisado mediante un enfoque conocido como aprendizaje semi-supervisado, el cual constituye una categoría intermedia dentro del aprendizaje automático. Esta técnica resulta especialmente útil cuando se dispone de una gran cantidad de datos sin etiquetar y solo una pequeña fracción del conjunto ha sido etiquetada manualmente, lo cual suele ser costoso y lento. En este contexto, los algoritmos no supervisados se utilizan primero para analizar y estructurar los datos sin etiquetar, identificando patrones o agrupaciones, y posteriormente se aplica aprendizaje supervisado sobre los datos que sí cuentan con etiquetas, lo cual mejora la precisión del modelo sin requerir el etiquetado completo de la base.

Este enfoque se ha aplicado exitosamente, por ejemplo, en tareas como la detección de fraude, donde se entrena primero con grandes volúmenes de transacciones no etiquetadas y luego con aquellas confirmadas como fraudulentas:

![img](https://d2908q01vomqb2.cloudfront.net/f1f836cb4ea6efb2a0b1b99f41ad8b103eff4b59/2023/04/13/ML-13908-label.jpg)

---

### Implementación en PySpark (MLlib)

**PySpark** es la interfaz de Python para Apache Spark y proporciona una poderosa plataforma distribuida para procesar grandes cantidades de datos. Dentro de PySpark, la biblioteca **MLlib** ofrece una variedad de algoritmos y herramientas para aplicar técnicas de aprendizaje automático en pipelines escalables y eficientes.

**Algoritmos supervisados en MLlib:**

* `LinearRegression`
* `LogisticRegression`
* `DecisionTreeClassifier`, `DecisionTreeRegressor`
* `RandomForestClassifier`, `RandomForestRegressor`
* `GBTClassifier` (Gradient Boosted Trees)

**Algoritmos no supervisados en MLlib:**

* `KMeans`
* `BisectingKMeans`
* `GaussianMixture`
* `PCA`

Además de los algoritmos, MLlib ofrece:

* **Pipelines** para estructurar flujos de trabajo (transformaciones + modelo).
* Herramientas para **selección y transformación de características**.
* **Evaluadores de modelos** con métricas como precisión, F1, área bajo la curva ROC, entre otros.

Gracias a su diseño distribuido, PySpark es ideal para proyectos de Big Data, donde se requiere procesar grandes volúmenes de información de manera paralela y eficiente.

---

### Referencias en formato APA

* Amazon Web Services. (n.d.). *What’s the Difference Between Supervised and Unsupervised Machine Learning?* Recuperado de [https://aws.amazon.com/compare/the-difference-between-machine-learning-supervised-and-unsupervised/](https://aws.amazon.com/compare/the-difference-between-machine-learning-supervised-and-unsupervised/)

* Google Cloud. (n.d.). *Supervised vs. unsupervised learning: What's the difference?* Recuperado de [https://cloud.google.com/discover/supervised-vs-unsupervised-learning](https://cloud.google.com/discover/supervised-vs-unsupervised-learning)

* Amazon Web Services. (n.d.). *Types of Algorithms - Amazon SageMaker*. Recuperado de [https://docs.aws.amazon.com/sagemaker/latest/dg/algorithms-choose.html](https://docs.aws.amazon.com/sagemaker/latest/dg/algorithms-choose.html)

* Apache Spark. (n.d.). *MLlib: Main Guide*. Recuperado de [https://spark.apache.org/docs/latest/ml-guide.html](https://spark.apache.org/docs/latest/ml-guide.html)


## 2. Selección de los datos
---

In [3]:
import findspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col, isnan, when, count
from pyspark.sql.types import StringType, DoubleType, FloatType
import findspark

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.clustering import GaussianMixture

from os import path

In [4]:
findspark.init()
findspark.find()

'/Users/cesarivp/Documents/GitHub/TC4034.10-Equipo-37/.venv/lib/python3.9/site-packages/pyspark'

In [5]:
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/25 21:17:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
PATH = "../files"
FILE = "amazon_electronics.csv"

In [7]:
class FileManager():
    @staticmethod
    def open_csv_file(input_path : str, file_name : str):
        """
        This method opens a csv file with pyspark.
        """
        csv_df = spark.read.csv(
            path.join(input_path, file_name),
            header=True,
            inferSchema=True,
            multiLine=True,
            escape="\"",
            quote="\""
        )

        csv_df.show(truncate=20)

        return csv_df

In [8]:
df_reviews = FileManager.open_csv_file(PATH, FILE)

+-----------+-----------+--------------+----------+--------------+--------------------+----------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+---------+
|marketplace|customer_id|     review_id|product_id|product_parent|       product_title|product_category|star_rating|helpful_votes|total_votes|vine|verified_purchase|     review_headline|         review_body|review_date|sentiment|
+-----------+-----------+--------------+----------+--------------+--------------------+----------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+---------+
|         US|   22873041|R3ARRMDEGED8RD|B00KJWQIIC|     335625766|Plemo 14-Inch Lap...|              PC|          5|            0|          0|   N|                Y|Pleasantly surprised|I was very surpri...| 2015-08-31|        1|
|         US|   30088427| RQ28TSA020Y6J|B013ALA9LA|     671157305|TP-Link OnHub 

Vamos a definir las columnas que serán útiles para el procesamiento comopara el entrenamiento del modelo.

In [9]:
RELEVANT_COLUMNS_FOR_CHARACTERIZATION = [
  "star_rating",
  "helpful_votes",
  "total_votes",
  "vine",
  "verified_purchase",
  "review_date",
  "sentiment"
]

In [10]:
df_reviews_filtered = df_reviews.select(*RELEVANT_COLUMNS_FOR_CHARACTERIZATION)

In [11]:
df_reviews_filtered.head(5)

[Row(star_rating=5, helpful_votes=0, total_votes=0, vine='N', verified_purchase='Y', review_date=datetime.date(2015, 8, 31), sentiment=1),
 Row(star_rating=5, helpful_votes=24, total_votes=31, vine='N', verified_purchase='N', review_date=datetime.date(2015, 8, 31), sentiment=1),
 Row(star_rating=1, helpful_votes=2, total_votes=2, vine='N', verified_purchase='N', review_date=datetime.date(2015, 8, 31), sentiment=0),
 Row(star_rating=1, helpful_votes=0, total_votes=0, vine='N', verified_purchase='Y', review_date=datetime.date(2015, 8, 31), sentiment=0),
 Row(star_rating=5, helpful_votes=0, total_votes=0, vine='N', verified_purchase='Y', review_date=datetime.date(2015, 8, 31), sentiment=1)]

## 3. Preparación de los datos
---

Se implementó un procedimiento automático en PySpark que:

1. Filtra los registros de la base de datos que cumplen con cada combinación de valores.
2. Almacena cada subconjunto en un diccionario indexado por nombre de combinación (ej. "R5_VPY_VN" para `star_rating`=5, `verified_purchase`=Y, `vine`=N).
3. Imprime la cantidad de registros por partición para control y trazabilidad.

Las particiones con muy pocos registros pueden ser descartadas en etapas posteriores para evitar problemas en el análisis.

In [12]:
class PartitioningManager:
    @staticmethod
    def compute_probabilities(df, cols):
        """
        Computes and returns the probability of each combination of values in the specified columns.
        """
        total_count = df.count()
        return df.groupBy(cols).count() \
                 .withColumn("probability", F.round(F.col("count") / total_count, 6)) \
                 .orderBy("probability", ascending=False)

    @staticmethod
    def filter_partition(df, star_rating, verified_purchase, vine):
        """
        Filters the DataFrame by specific values for rating, verified purchase, and vine.
        """
        return df.filter(
            (F.col("star_rating") == star_rating) &
            (F.col("verified_purchase") == verified_purchase) &
            (F.col("vine") == vine)
        )

    @staticmethod
    def generate_all_partitions(df, min_probability=0.0001):
        """
        Generates partitions only for combinations whose joint probability is above min_probability.
        """

        prob_df = PartitioningManager.compute_probabilities(
            df, ["star_rating", "verified_purchase", "vine"]
        )

        filtered_combinations = prob_df.filter(
            F.col("probability") >= min_probability
        ).select("star_rating", "verified_purchase", "vine").collect()

        partitions = {}
        for row in filtered_combinations:
            rating = row["star_rating"]
            purchase = row["verified_purchase"]
            vine = row["vine"]

            key = f"R{rating}_VP{purchase}_V{vine}"
            filtered = PartitioningManager.filter_partition(df, rating, purchase, vine)
            partitions[key] = filtered
            print(f"Partition {key} created with {filtered.count()} records.")

        return partitions

    @staticmethod
    def stratified_sample_partitioned_data(partitions_dict, label_col="sentiment", fraction=0.3, min_rows=50):
        """
        Applies stratified sampling to each partition based on sentiment.
        """
        sampled_partitions = {}

        for key, df in partitions_dict.items():
            count = df.count()

            if count < min_rows:
                print(f"Skipping partition {key} — only {count} rows (<{min_rows})")
                continue

            sentiments = df.select(label_col).distinct().rdd.flatMap(lambda x: x).collect()
            fractions = {s: fraction for s in sentiments}

            sampled_df = df.sampleBy(label_col, fractions, seed=42)
            sampled_partitions[key] = sampled_df
            print(f"Sampled {sampled_df.count()} rows from partition {key} (original: {count})")

        return sampled_partitions

    @staticmethod
    def build_combined_sample(partitions_sampled_dict):
        """
        Unites all sampled partitions into a single DataFrame (M).
        This helps reduce computational load while maintaining diversity.
        """
        if not partitions_sampled_dict:
            raise ValueError("No partitions provided for sample combination.")

        combined_df = None
        for key, df in partitions_sampled_dict.items():
            if combined_df is None:
                combined_df = df
            else:
                combined_df = combined_df.union(df)
            print(f"Partition {key} added to the combined sample.")

        print(f"Total records in combined sample: {combined_df.count()}")
        return combined_df


In [13]:
partitions = PartitioningManager.generate_all_partitions(df_reviews, min_probability=0.00001)

25/05/25 21:17:35 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Partition R5_VPY_VN created with 3679909 records.


Partition R4_VPY_VN created with 1019728 records.


Partition R1_VPY_VN created with 603371 records.


Partition R3_VPY_VN created with 443364 records.


...

Partition R4_VPN_VY created with 13240 records.


Partition R3_VPN_VY created with 4886 records.


Partition R2_VPN_VY created with 1634 records.


Partition R1_VPN_VY created with 705 records.


Partition R5_VPY_VY created with 101 records.


In [14]:
# Sample of one of the generated partitions.
partitions["R4_VPY_VN"].show(5)

+-----------+-----------+--------------+----------+--------------+--------------------+----------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+---------+
|marketplace|customer_id|     review_id|product_id|product_parent|       product_title|product_category|star_rating|helpful_votes|total_votes|vine|verified_purchase|     review_headline|         review_body|review_date|sentiment|
+-----------+-----------+--------------+----------+--------------+--------------------+----------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+---------+
|         US|   49329488|R1QF6RS1PDLU18|B00TR05L9Y|     778403103|Lenovo TAB2 A10 -...|              PC|          4|            1|          1|   N|                Y|                Good|I am not sure I d...| 2015-08-31|        1|
|         US|   43341796|R2NJ3WFUS4E5G6|B00YGJJQ6U|     986548413|Fintie iPad Ai

### Técnica de muestreo aplicada por partición

Una vez construidas las particiones, se aplicó una técnica de **muestreo estratificado** sobre cada subconjunto, usando la variable `sentiment` como variable de estratificación. Esto asegura que cada muestra mantenga la proporción original de clases de sentimiento en la partición.

Para evitar particiones con tamaños insuficientes, se definió un umbral mínimo (`min_rows`) que descarta automáticamente las particiones con muy pocos registros.

Además, se permitió configurar el porcentaje de muestreo (`fraction`) por clase de sentimiento. Esto ofrece flexibilidad para ajustar el tamaño del conjunto de entrenamiento o validación según necesidades posteriores.

#### Justificación del muestreo estratificado

* **Preservación del equilibrio de clases**: Al muestrear por clase de sentimiento se evitan sesgos por clases desbalanceadas.
* **Relevancia contextual**: Al aplicar el muestreo dentro de cada partición (y no sobre la base completa), se conserva la variabilidad contextual de las reseñas.
* **Evita el submuestreo accidental**: Las particiones muy pequeñas son descartadas de forma controlada, garantizando que el conjunto final tenga representatividad suficiente.

In [15]:
sampled_partitions = PartitioningManager.stratified_sample_partitioned_data(partitions, fraction=0.05, min_rows=100)

Sampled 184082 rows from partition R5_VPY_VN (original: 3679909)


Sampled 51148 rows from partition R4_VPY_VN (original: 1019728)


Sampled 30183 rows from partition R1_VPY_VN (original: 603371)


Sampled 22223 rows from partition R3_VPY_VN (original: 443364)


Sampled 20507 rows from partition R5_VPN_VN (original: 410073)


...

Sampled 724 rows from partition R4_VPN_VY (original: 13240)


Sampled 266 rows from partition R3_VPN_VY (original: 4886)


Sampled 93 rows from partition R2_VPN_VY (original: 1634)


Sampled 35 rows from partition R1_VPN_VY (original: 705)


Sampled 3 rows from partition R5_VPY_VY (original: 101)


In [16]:
df_sample_M = PartitioningManager.build_combined_sample(sampled_partitions)

Partition R5_VPY_VN added to the combined sample.
Partition R4_VPY_VN added to the combined sample.
Partition R1_VPY_VN added to the combined sample.
Partition R3_VPY_VN added to the combined sample.
Partition R5_VPN_VN added to the combined sample.
Partition R2_VPY_VN added to the combined sample.
Partition R1_VPN_VN added to the combined sample.
Partition R4_VPN_VN added to the combined sample.
Partition R3_VPN_VN added to the combined sample.
Partition R2_VPN_VN added to the combined sample.
Partition R5_VPN_VY added to the combined sample.
Partition R4_VPN_VY added to the combined sample.
Partition R3_VPN_VY added to the combined sample.
Partition R2_VPN_VY added to the combined sample.
Partition R1_VPN_VY added to the combined sample.
Partition R5_VPY_VY added to the combined sample.


Total records in combined sample: 345829


Ya que tenemos listos las particiones, podemos empezar a analizar el dataset para ver si tenemos que realizar algún tipo de pre-procesamiento antes de entrenar los modelos.

In [17]:
class StatisticalAnalysisHelper():
    @staticmethod
    def dataset_dimensions(df_input):
        print("columns in the dataset:", len(df_input.columns))
        print("rows in the dataset:", df_input.count())

    @staticmethod
    def schema_information(df_input):
        """
        This method shows the current schema of the data.
        """
        df_input.printSchema()

    @staticmethod
    def descriptive_statistics(df_input):
        """
        This method shows the descriptive statistics of the data.
        """
        df_input.summary().show(truncate=False)

    @staticmethod
    def missing_values_table(df_input):
        """
        Displays a table with the count of missing values per column.
        """
        missing_exprs = []
        
        for c in df_input.schema.fields:
            field_name = c.name
            field_type = c.dataType
            
            if isinstance(field_type, (DoubleType, FloatType)):
                missing_exprs.append(
                    count(when(col(field_name).isNull() | isnan(col(field_name)), field_name)).alias(field_name)
                )
            elif isinstance(field_type, StringType):
                missing_exprs.append(
                    count(when(col(field_name).isNull() | (col(field_name) == ""), field_name)).alias(field_name)
                )
            else:
                missing_exprs.append(
                    count(when(col(field_name).isNull(), field_name)).alias(field_name)
                )

        df_missing_values = df_input.select(missing_exprs)

        return df_missing_values

In [18]:
StatisticalAnalysisHelper.dataset_dimensions(df_sample_M)

columns in the dataset: 16


rows in the dataset: 345829


In [19]:
StatisticalAnalysisHelper.schema_information(df_sample_M)

root
 |-- marketplace: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- review_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_parent: integer (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- star_rating: integer (nullable = true)
 |-- helpful_votes: integer (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- vine: string (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- sentiment: integer (nullable = true)



In [20]:
StatisticalAnalysisHelper.descriptive_statistics(df_sample_M)

25/05/25 21:26:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------+--------------------+--------------+-------------------+--------------------+----------------------------------------------------------------------------------------------+----------------+------------------+------------------+------------------+------+-----------------+----------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+
|summary|marketplace|customer_id         |review_id     |product_id         |product_parent      |product_title                                  

In [21]:
missing_values = StatisticalAnalysisHelper.missing_values_table(df_sample_M)
missing_values.show(truncate=False)

+-----------+-----------+---------+----------+--------------+-------------+----------------+-----------+-------------+-----------+----+-----------------+---------------+-----------+-----------+---------+
|marketplace|customer_id|review_id|product_id|product_parent|product_title|product_category|star_rating|helpful_votes|total_votes|vine|verified_purchase|review_headline|review_body|review_date|sentiment|
+-----------+-----------+---------+----------+--------------+-------------+----------------+-----------+-------------+-----------+----+-----------------+---------------+-----------+-----------+---------+
|0          |0          |0        |0         |0             |0            |0               |0          |0            |0          |0   |0                |0              |0          |0          |0        |
+-----------+-----------+---------+----------+--------------+-------------+----------------+-----------+-------------+-----------+----+-----------------+---------------+-----------+---

Como podemos observar, el conjunto de datos no tiene datos vacíos y parece ya haber pasado por algún tipo de limpieza antes de subirse al repositorio público, por lo que no tenemos que hacer nada para que el modelo funcione de manera óptima.

## 4. Preparación del conjunto de entrenamiento y prueba
--- 

In [22]:
class TrainTestManager:
  @staticmethod
  def stratified_train_test_split(df, label_col="sentiment", train_ratio=0.8, seed=42):
    """
    Performs a stratified split of the DataFrame based on the label column.
    Returns (train_df, test_df).
    """
    label_values = df.select(label_col).distinct().rdd.flatMap(lambda x: x).collect()

    train_fractions = {label: train_ratio for label in label_values}

    train_df = df.sampleBy(label_col, train_fractions, seed=seed)

    train_ids = train_df.select(F.monotonically_increasing_id().alias("id"))
    df_with_id = df.withColumn("id", F.monotonically_increasing_id())

    test_df = df_with_id.join(train_ids, on="id", how="left_anti").drop("id")
    train_df = train_df.drop("id") if "id" in train_df.columns else train_df

    return train_df, test_df

In [23]:
train_df, test_df = TrainTestManager.stratified_train_test_split(df_sample_M, train_ratio=0.8)

In [24]:
print(f"Train set: {train_df.count()} rows")
print(f"Test set: {test_df.count()} rows")

Train set: 276731 rows


Test set: 69098 rows


## 5. Construcción de modelos de aprendizaje supervisado y no supervisado
---

Para esta actividad, vamos a usar para como modelo supervisado la clase `Random Forest Classifier` y para el modelo supervisado `Gaussian Mixture`.

In [25]:
class ModelManager:
    @staticmethod
    def train_random_forest(train_df, test_df, label_col="sentiment", num_trees=50):
        """
        Trains a Random Forest classifier on the training set and evaluates it.
        Returns the trained model, accuracy score, and predictions.
        """
        categorical_cols = ["verified_purchase", "vine"]
        numeric_cols = ["helpful_votes", "total_votes"]

        indexers = [
            StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
            for col in categorical_cols
        ]

        feature_cols = [f"{col}_idx" for col in categorical_cols] + numeric_cols
        assembler = VectorAssembler(inputCols=feature_cols, outputCol="assembled_features")

        rf = RandomForestClassifier(
            labelCol=label_col,
            featuresCol="assembled_features",
            numTrees=num_trees,
            maxDepth=10,
            seed=42
        )

        pipeline = Pipeline(stages=indexers + [assembler, rf])
        model = pipeline.fit(train_df)

        predictions = model.transform(test_df)

        evaluator = MulticlassClassificationEvaluator(
            labelCol=label_col,
            predictionCol="prediction",
            metricName="accuracy"
        )

        accuracy = evaluator.evaluate(predictions)

        return model, accuracy, predictions

    @staticmethod
    def train_gmm(train_df, test_df, k=2):
        """
        Trains a Gaussian Mixture Model on the training set and evaluates it on the test set.
        Returns the trained model and the silhouette score (as accuracy).
        """
        categorical_cols = ["verified_purchase", "vine"]
        numeric_cols = ["helpful_votes", "total_votes"]

        indexers = [
            StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
            for col in categorical_cols
        ]

        feature_cols = [f"{col}_idx" for col in categorical_cols] + numeric_cols
        assembler = VectorAssembler(inputCols=feature_cols, outputCol="assembled_features")
        scaler = StandardScaler(inputCol="assembled_features", outputCol="features")

        gmm = GaussianMixture(featuresCol="features", predictionCol="cluster", k=k, seed=42)

        pipeline = Pipeline(stages=indexers + [assembler, scaler, gmm])
        model = pipeline.fit(train_df)

        test_predictions = model.transform(test_df)

        evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="cluster", metricName="silhouette")
        accuracy = evaluator.evaluate(test_predictions)

        return model, accuracy

### Resultados del modelo supervisado `RandomForestClassifier`

In [26]:
rf_model, rf_accuracy, rf_predictions = ModelManager.train_random_forest(train_df, test_df)

25/05/25 21:32:00 WARN DAGScheduler: Broadcasting large task binary with size 1179.1 KiB


In [27]:
print(f"Random Forest accuracy: {round(rf_accuracy, 4)}")

Random Forest accuracy: 0.785


In [28]:
rf_model.transform(test_df).select("review_id", "prediction", "probability", "sentiment").show(10)

+--------------+----------+--------------------+---------+
|     review_id|prediction|         probability|sentiment|
+--------------+----------+--------------------+---------+
|R38YS6F8AIPTWS|       1.0|[0.19225900782666...|        1|
| RLKRLZ7VRDTGD|       1.0|[0.32355523152611...|        1|
|R3CMHUMRLBFRGU|       1.0|[0.19225900782666...|        1|
| RL5A9G7ATCTKW|       1.0|[0.19225900782666...|        1|
|R2WRS7LOHTEWC1|       1.0|[0.42124859352533...|        1|
|R30FVKNS7815YT|       1.0|[0.19225900782666...|        1|
| RIG8ZUVB49HVN|       1.0|[0.20638002970065...|        1|
|R2JTOYUS285KME|       1.0|[0.19225900782666...|        1|
| RRFWALHZISB9T|       1.0|[0.31802340260685...|        1|
|R39DPWJB2WS35W|       1.0|[0.19225900782666...|        1|
+--------------+----------+--------------------+---------+
only showing top 10 rows



### Resultados del modelo no supervisado `GaussianMixture`

In [29]:
gmm_model, gmm_accuracy = ModelManager.train_gmm(train_df, test_df)

25/05/25 21:35:20 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/05/25 21:35:20 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
25/05/25 21:35:21 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [30]:
print(f"Gaussian Mixture accuracy: {round(gmm_accuracy, 4)}")

Gaussian Mixture accuracy: 0.9557


In [31]:
gmm_model.transform(test_df).select("review_id", "cluster", "probability", "sentiment").show(10)

+--------------+-------+--------------------+---------+
|     review_id|cluster|         probability|sentiment|
+--------------+-------+--------------------+---------+
|R38YS6F8AIPTWS|      1|[6.72142143506112...|        1|
| RLKRLZ7VRDTGD|      1|[8.08858548615171...|        1|
|R3CMHUMRLBFRGU|      1|[6.72142143506112...|        1|
| RL5A9G7ATCTKW|      1|[6.72142143506112...|        1|
|R2WRS7LOHTEWC1|      1|[1.44286544793165...|        1|
|R30FVKNS7815YT|      1|[6.72142143506112...|        1|
| RIG8ZUVB49HVN|      1|[6.74562608221093...|        1|
|R2JTOYUS285KME|      1|[6.72142143506112...|        1|
| RRFWALHZISB9T|      1|[8.02229463127180...|        1|
|R39DPWJB2WS35W|      1|[6.72142143506112...|        1|
+--------------+-------+--------------------+---------+
only showing top 10 rows

